# H100 Heavy Experiment 04 — MultiNLI Genre Transfer and Bias

393k-row MultiNLI task. Domains are training genres such as fiction, government, and telephone; evaluation includes matched/mismatched style effects.

This notebook keeps the previous B0 → domain-specific fine-tuning → evaluation → bias recommendation timeline, but uses larger H100-grade models and datasets.

In [1]:
# Colab A100/H100 environment setup. Run this once, then restart the runtime.
import sys, subprocess, pathlib
req = pathlib.Path('requirements-colab.txt')
if req.exists():
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '-r', str(req)]
else:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2', 'pandas==2.2.2',
        'matplotlib>=3.8,<4', 'transformers>=4.45,<5', 'datasets>=3.0,<5',
        'accelerate>=1.0,<2', 'peft>=0.14,<1', 'sentencepiece', 'protobuf', 'emoji==0.6.0']
subprocess.check_call(cmd)
print('Environment installed. Restart the Colab runtime now, then run from the next cell.')


Environment ready. If package versions changed substantially, restart the kernel and rerun.


## Experiment configuration
Change `run_profile` to `h100_probe`, `h100_long`, `h100_extreme`, or for Longformer `h100_long_context`.

In [2]:
EXPERIMENT_SPEC = {
  "filename": "13_mnli_genre_nli_bias.ipynb",
  "notebook_id": "13_mnli_genre_nli_bias",
  "title": "H100 Heavy Experiment 04 — MultiNLI Genre Transfer and Bias",
  "task_kind": "mnli_genre",
  "models": [
    "microsoft/deberta-v3-large",
    "FacebookAI/roberta-large"
  ],
  "run_profile": "h100_long",
  "target_bias_score": 0.04,
  "description": "393k-row MultiNLI task. Domains are training genres such as fiction, government, and telephone; evaluation includes matched/mismatched style effects.",
  "require_high_end_gpu": True,
  "reset_output_dir": True,
  "gradient_checkpointing": True
}

## Training progress visualization added
This version records every Hugging Face `Trainer.state.log_history` row, saves per-run training logs, and exports learning-curve figures for training loss, validation loss, validation F1, accuracy, and learning rate.
After execution, check `tables/training_history.csv`, `tables/training_diagnostics.csv`, and the `figures/` folder.


In [3]:

try:
    EXPERIMENT_SPEC
except NameError:
    EXPERIMENT_SPEC = {}
EXPERIMENT_SPEC = dict(EXPERIMENT_SPEC or {})
EXPERIMENT_SPEC.setdefault('filename', '10_civil_comments_identity_bias.ipynb')
EXPERIMENT_SPEC.setdefault('notebook_id', 'bertweet_civil_bias_notebook')
EXPERIMENT_SPEC.setdefault('title', 'BERTweet CivilComments Bias Probe')
EXPERIMENT_SPEC.setdefault('task_kind', 'civil_toxicity_identity')
EXPERIMENT_SPEC.setdefault('models', ['vinai/bertweet-base'])
EXPERIMENT_SPEC.setdefault('run_profile', 'h100_probe')
EXPERIMENT_SPEC.setdefault('target_bias_score', 0.05)
EXPERIMENT_SPEC.setdefault('require_high_end_gpu', True)
EXPERIMENT_SPEC.setdefault('reset_output_dir', False)
EXPERIMENT_SPEC.setdefault('gradient_checkpointing', True)
EXPERIMENT_SPEC.setdefault('auto_scale_batch_size', False)
EXPERIMENT_SPEC.setdefault('skip_failed_runs', False)
EXPERIMENT_SPEC.setdefault('profile_overrides', {})
import os, sys, subprocess, json, math, random, time, gc, inspect, shutil, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('WANDB_DISABLED', 'true')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model, TaskType
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame, is_cuda_oom_like, format_exception
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
    def is_cuda_oom_like(exc):
        text=f'{type(exc).__name__}: {exc}'.lower()
        return any(x in text for x in ['cuda out of memory','outofmemoryerror','acceleratorerror','cuda error','layer_norm'])
    def format_exception(exc):
        import traceback
        return ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
try:
    from aix_bias_suite.runtime import safe_enable_gradient_checkpointing, validate_non_empty_training_frame
except Exception:
    def safe_enable_gradient_checkpointing(model):
        if hasattr(model, 'config') and hasattr(model.config, 'use_cache'):
            model.config.use_cache = False
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        except TypeError:
            model.gradient_checkpointing_enable()
        except Exception:
            pass
        if hasattr(model, 'enable_input_require_grads'):
            try: model.enable_input_require_grads()
            except Exception: pass
    def validate_non_empty_training_frame(train_df, run_id):
        if len(train_df) == 0:
            raise ValueError(f'{run_id} has 0 training rows. Check domain splitting.')
try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# Safety / runtime setup
# ============================================================
SEED = EXPERIMENT_SPEC.get('seed', 42)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_PROPS = torch.cuda.get_device_properties(0)
    GPU_MEMORY_GB = GPU_PROPS.total_memory / (1024 ** 3)
    CUDA_CC = f'{GPU_PROPS.major}.{GPU_PROPS.minor}'
else:
    GPU_NAME = 'CPU'
    GPU_MEMORY_GB = 0.0
    CUDA_CC = '0.0'
print('Detected device:', GPU_NAME)
print(f'GPU memory: {GPU_MEMORY_GB:.1f} GB')
print('Compute capability:', CUDA_CC)

GPU_NAME_UP = GPU_NAME.upper()
IS_HIGH_END_GPU = bool(torch.cuda.is_available() and (
    'H100' in GPU_NAME_UP or 'A100' in GPU_NAME_UP or 'BLACKWELL' in GPU_NAME_UP or GPU_MEMORY_GB >= 40
))
if EXPERIMENT_SPEC.get('require_high_end_gpu', True) and not IS_HIGH_END_GPU:
    raise RuntimeError(f'This notebook is designed for H100/A100/Blackwell or >=40GB VRAM GPU. Current device={GPU_NAME}, memory={GPU_MEMORY_GB:.1f} GB')

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)
print('USE_BF16:', USE_BF16, 'USE_FP16:', USE_FP16)

# ============================================================
# Profiles: probe / long / extreme
# ============================================================
RUN_PROFILE = str(EXPERIMENT_SPEC.get('run_profile', 'h100_probe'))
PROFILE_TABLE = {
    'h100_probe': {
        'epochs': 1, 'max_length': 256, 'batch_size': 16, 'eval_batch_size': 64,
        'max_train_per_domain': 10000, 'max_eval_per_set': 2000, 'baseline_max_each_domain': 10000,
        'lora_ranks': [8], 'adapter_bottlenecks': [64], 'do_full_ft': False,
        'gradient_accumulation_steps': 1, 'dataloader_num_workers': 2,
    },
    'h100_long': {
        'epochs': 3, 'max_length': 512, 'batch_size': 32, 'eval_batch_size': 128,
        'max_train_per_domain': 120000, 'max_eval_per_set': 20000, 'baseline_max_each_domain': 120000,
        'lora_ranks': [8, 16, 32], 'adapter_bottlenecks': [64, 128], 'do_full_ft': True,
        'gradient_accumulation_steps': 1, 'dataloader_num_workers': 4,
    },
    'h100_extreme': {
        'epochs': 4, 'max_length': 512, 'batch_size': 32, 'eval_batch_size': 128,
        'max_train_per_domain': None, 'max_eval_per_set': None, 'baseline_max_each_domain': None,
        'lora_ranks': [8, 16, 32, 64], 'adapter_bottlenecks': [64, 128, 256], 'do_full_ft': True,
        'gradient_accumulation_steps': 1, 'dataloader_num_workers': 4,
    },
    'h100_long_context': {
        'epochs': 2, 'max_length': 2048, 'batch_size': 2, 'eval_batch_size': 8,
        'max_train_per_domain': 60000, 'max_eval_per_set': 10000, 'baseline_max_each_domain': 60000,
        'lora_ranks': [8, 16], 'adapter_bottlenecks': [64, 128], 'do_full_ft': False,
        'gradient_accumulation_steps': 8, 'dataloader_num_workers': 4,
    },
}
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
if RUN_PROFILE not in PROFILE_TABLE:
    print(f"Unknown run_profile={RUN_PROFILE!r}; falling back to 'h100_probe'. Valid profiles={list(PROFILE_TABLE)}")
    RUN_PROFILE = 'h100_probe'
PROFILE = PROFILE_TABLE[RUN_PROFILE].copy()
PROFILE.update(EXPERIMENT_SPEC.get('profile_overrides', {}))
if EXPERIMENT_SPEC.get('auto_scale_batch_size', False) and GPU_MEMORY_GB >= 80 and RUN_PROFILE not in ['h100_long_context']:
    PROFILE['batch_size'] = max(PROFILE['batch_size'], 64 if RUN_PROFILE != 'h100_probe' else 32)
    PROFILE['eval_batch_size'] = max(PROFILE['eval_batch_size'], 256 if RUN_PROFILE != 'h100_probe' else 128)
print('RUN_PROFILE:', RUN_PROFILE)
print('PROFILE:', PROFILE)

TEXT_COL='text'; TEXT_PAIR_COL='text_pair'; LABEL_COL='label'; DOMAIN_COL='domain'
GROUP_COL='demographic_group'; PERTURB_COL='perturbation_type'; SOURCE_ID_COL='source_id'; IDENTITY_COL='identity_term'
NOTEBOOK_ID = EXPERIMENT_SPEC.get('notebook_id') or 'bertweet_civil_bias_notebook'
OUT_ROOT = Path(EXPERIMENT_SPEC.get('output_root') or f"/content/{NOTEBOOK_ID}_outputs")
TABLE_ROOT = OUT_ROOT / 'tables'; METRIC_ROOT = OUT_ROOT / 'metrics'; CKPT_ROOT = OUT_ROOT / 'checkpoints'; RUN_ROOT = OUT_ROOT / 'runs'; LOG_ROOT = OUT_ROOT / 'training_logs'; FIG_ROOT = OUT_ROOT / 'figures'
if EXPERIMENT_SPEC.get('reset_output_dir', True) and OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
for p in [OUT_ROOT, TABLE_ROOT, METRIC_ROOT, CKPT_ROOT, RUN_ROOT, LOG_ROOT, FIG_ROOT]: p.mkdir(parents=True, exist_ok=True)

# ============================================================
# General helpers
# ============================================================
def safe_display(df, n=20):
    try:
        display(df.head(n) if hasattr(df, 'head') else df)
    except Exception:
        print(df.head(n).to_string() if hasattr(df, 'head') else df)

def limit_df(df, max_n=None, seed=SEED, stratify_col=LABEL_COL):
    df = df.reset_index(drop=True)
    if max_n is None or len(df) <= max_n:
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    stratify = None
    if stratify_col in df.columns:
        counts = df[stratify_col].value_counts()
        if len(counts) > 1 and counts.min() >= 2:
            stratify = df[stratify_col]
    sample, _ = train_test_split(df, train_size=max_n, random_state=seed, stratify=stratify)
    return sample.sample(frac=1.0, random_state=seed).reset_index(drop=True)

def split_train_valid_test(df, seed=SEED, valid_size=0.1, test_size=0.1):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    strat = df[LABEL_COL] if df[LABEL_COL].value_counts().min() >= 2 else None
    train, temp = train_test_split(df, test_size=valid_size+test_size, random_state=seed, stratify=strat)
    strat2 = temp[LABEL_COL] if temp[LABEL_COL].value_counts().min() >= 2 else None
    valid, test = train_test_split(temp, test_size=test_size/(valid_size+test_size), random_state=seed, stratify=strat2)
    return train.reset_index(drop=True), valid.reset_index(drop=True), test.reset_index(drop=True)

def find_col(cols, cands):
    for c in cands:
        if c in cols: return c
    return None

def classlabel_name(feature, value):
    try:
        if hasattr(feature, 'int2str'):
            return feature.int2str(int(value))
        if hasattr(feature, 'names'):
            return feature.names[int(value)]
    except Exception:
        pass
    return str(value)

# ============================================================
# Training log / loss visualization helpers
# ============================================================
def _safe_run_filename(run_id):
    return str(run_id).replace('/', '__').replace(' ', '_').replace(':', '_')


def extract_training_log_df(trainer, run_meta):
    """Convert Hugging Face Trainer log_history into a tidy DataFrame."""
    rows = []
    for i, log in enumerate(getattr(trainer.state, 'log_history', [])):
        row = dict(run_meta)
        row['log_index'] = i
        for k, v in log.items():
            if isinstance(v, (np.integer,)):
                row[k] = int(v)
            elif isinstance(v, (np.floating,)):
                row[k] = float(v)
            elif isinstance(v, (int, float, str, bool)) or v is None:
                row[k] = v
            else:
                row[k] = str(v)
        if 'loss' in row:
            row['train_loss'] = row.get('loss')
        if 'eval_loss' in row:
            row['validation_loss'] = row.get('eval_loss')
        if 'eval_f1_macro' in row:
            row['validation_f1_macro'] = row.get('eval_f1_macro')
        if 'eval_accuracy' in row:
            row['validation_accuracy'] = row.get('eval_accuracy')
        rows.append(row)
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    for col in ['step','epoch','loss','train_loss','eval_loss','validation_loss','eval_f1_macro','validation_f1_macro','eval_accuracy','validation_accuracy','learning_rate']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def summarize_training_log_df(log_df):
    """Create compact learning-progress diagnostics from a single run log."""
    summary = {
        'train_log_points': 0,
        'eval_log_points': 0,
        'first_train_loss': np.nan,
        'last_train_loss': np.nan,
        'min_train_loss': np.nan,
        'train_loss_delta': np.nan,
        'train_loss_reduction_pct': np.nan,
        'first_eval_loss': np.nan,
        'last_eval_loss': np.nan,
        'min_eval_loss': np.nan,
        'eval_loss_delta': np.nan,
        'eval_loss_reduction_pct': np.nan,
        'best_eval_f1_macro_from_log': np.nan,
        'best_eval_accuracy_from_log': np.nan,
        'best_eval_step_from_log': np.nan,
        'best_eval_epoch_from_log': np.nan,
        'final_learning_rate': np.nan,
        'loss_generalization_gap_last': np.nan,
        'overfit_warning_by_loss': False,
    }
    if log_df is None or len(log_df) == 0:
        return summary

    train = pd.DataFrame()
    if 'train_loss' in log_df.columns:
        train = log_df[log_df['train_loss'].notna()].copy()
    evals = pd.DataFrame()
    if 'validation_loss' in log_df.columns:
        evals = log_df[log_df['validation_loss'].notna()].copy()

    summary['train_log_points'] = int(len(train))
    summary['eval_log_points'] = int(len(evals))

    if len(train):
        first = float(train['train_loss'].iloc[0])
        last = float(train['train_loss'].iloc[-1])
        mn = float(train['train_loss'].min())
        summary.update({
            'first_train_loss': first,
            'last_train_loss': last,
            'min_train_loss': mn,
            'train_loss_delta': last - first,
            'train_loss_reduction_pct': ((first - last) / first * 100.0) if first != 0 else np.nan,
        })
        if 'learning_rate' in train.columns and train['learning_rate'].notna().any():
            summary['final_learning_rate'] = float(train['learning_rate'].dropna().iloc[-1])

    if len(evals):
        first = float(evals['validation_loss'].iloc[0])
        last = float(evals['validation_loss'].iloc[-1])
        mn = float(evals['validation_loss'].min())
        summary.update({
            'first_eval_loss': first,
            'last_eval_loss': last,
            'min_eval_loss': mn,
            'eval_loss_delta': last - first,
            'eval_loss_reduction_pct': ((first - last) / first * 100.0) if first != 0 else np.nan,
        })
        if 'validation_f1_macro' in evals.columns and evals['validation_f1_macro'].notna().any():
            best_idx = evals['validation_f1_macro'].idxmax()
            summary['best_eval_f1_macro_from_log'] = float(evals.loc[best_idx, 'validation_f1_macro'])
            summary['best_eval_step_from_log'] = float(evals.loc[best_idx, 'step']) if 'step' in evals.columns and pd.notna(evals.loc[best_idx, 'step']) else np.nan
            summary['best_eval_epoch_from_log'] = float(evals.loc[best_idx, 'epoch']) if 'epoch' in evals.columns and pd.notna(evals.loc[best_idx, 'epoch']) else np.nan
        if 'validation_accuracy' in evals.columns and evals['validation_accuracy'].notna().any():
            summary['best_eval_accuracy_from_log'] = float(evals['validation_accuracy'].max())

    if len(train) and len(evals):
        gap = summary['last_eval_loss'] - summary['last_train_loss']
        summary['loss_generalization_gap_last'] = float(gap)
        # A rough warning only: validation loss worse than train loss by a visible margin and not improving.
        summary['overfit_warning_by_loss'] = bool(gap > 0.25 and summary.get('eval_loss_delta', 0) > 0)

    return summary


def plot_training_diagnostics(training_history_df, training_diag_df=None, ranking_df=None):
    """Save and display loss/metric curves that show how much each run learned."""
    FIG_ROOT.mkdir(parents=True, exist_ok=True)
    saved_paths = []

    if training_history_df is None or len(training_history_df) == 0:
        print('[Training diagnostics] No Trainer log_history was collected.')
        return saved_paths

    df = training_history_df.copy()
    if 'step' not in df.columns:
        df['step'] = np.nan
    if 'epoch' not in df.columns:
        df['epoch'] = np.nan

    # Focus plots on top-ranked runs if available; otherwise plot all runs.
    top_runs = None
    if ranking_df is not None and len(ranking_df) and 'run_id' in ranking_df.columns:
        top_runs = ranking_df.head(12)['run_id'].astype(str).tolist()
        plot_df = df[df['run_id'].astype(str).isin(top_runs)].copy()
        if len(plot_df) == 0:
            plot_df = df.copy()
    else:
        plot_df = df.copy()

    def _plot_line(metric_col, title, ylabel, filename):
        if metric_col not in plot_df.columns or plot_df[metric_col].dropna().empty:
            return None
        plt.figure(figsize=(14, 7))
        for run_id, sub in plot_df[plot_df[metric_col].notna()].groupby('run_id'):
            sub = sub.sort_values(['step','epoch','log_index'])
            x = sub['step'] if sub['step'].notna().any() else np.arange(len(sub))
            plt.plot(x, sub[metric_col], marker='o', linewidth=1.5, markersize=3, label=str(run_id)[:70])
        plt.title(title)
        plt.xlabel('Global step')
        plt.ylabel(ylabel)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=7, loc='best')
        plt.tight_layout()
        out = FIG_ROOT / filename
        plt.savefig(out, dpi=180, bbox_inches='tight')
        plt.show()
        plt.close()
        return str(out)

    for metric_col, title, ylabel, filename in [
        ('train_loss', 'Training Loss by Run', 'Training loss', 'training_loss_by_run.png'),
        ('validation_loss', 'Validation Loss by Run', 'Validation loss', 'validation_loss_by_run.png'),
        ('validation_f1_macro', 'Validation F1 Macro by Run', 'Validation F1 macro', 'validation_f1_macro_by_run.png'),
        ('validation_accuracy', 'Validation Accuracy by Run', 'Validation accuracy', 'validation_accuracy_by_run.png'),
        ('learning_rate', 'Learning Rate Schedule by Run', 'Learning rate', 'learning_rate_by_run.png'),
    ]:
        p = _plot_line(metric_col, title, ylabel, filename)
        if p: saved_paths.append(p)

    if training_diag_df is not None and len(training_diag_df):
        diag = training_diag_df.copy()
        # Save a bar chart for loss reduction. Positive means loss decreased.
        if 'train_loss_reduction_pct' in diag.columns and diag['train_loss_reduction_pct'].notna().any():
            diag_plot = diag.sort_values('train_loss_reduction_pct', ascending=True).tail(20)
            plt.figure(figsize=(12, max(5, 0.35 * len(diag_plot))))
            plt.barh(diag_plot['run_id'].astype(str), diag_plot['train_loss_reduction_pct'])
            plt.title('Top Training Loss Reduction (%)')
            plt.xlabel('Loss reduction %: (first_loss - last_loss) / first_loss')
            plt.ylabel('Run ID')
            plt.grid(True, axis='x', alpha=0.3)
            plt.tight_layout()
            out = FIG_ROOT / 'top_training_loss_reduction_pct.png'
            plt.savefig(out, dpi=180, bbox_inches='tight')
            plt.show()
            plt.close()
            saved_paths.append(str(out))

        if 'loss_generalization_gap_last' in diag.columns and diag['loss_generalization_gap_last'].notna().any():
            diag_plot = diag.sort_values('loss_generalization_gap_last', ascending=False).head(20)
            plt.figure(figsize=(12, max(5, 0.35 * len(diag_plot))))
            plt.barh(diag_plot['run_id'].astype(str), diag_plot['loss_generalization_gap_last'])
            plt.title('Largest Final Loss Generalization Gap')
            plt.xlabel('last_eval_loss - last_train_loss')
            plt.ylabel('Run ID')
            plt.gca().invert_yaxis()
            plt.grid(True, axis='x', alpha=0.3)
            plt.tight_layout()
            out = FIG_ROOT / 'loss_generalization_gap_top20.png'
            plt.savefig(out, dpi=180, bbox_inches='tight')
            plt.show()
            plt.close()
            saved_paths.append(str(out))

    return saved_paths

# ============================================================
# Dataset builders
# ============================================================
CIVIL_IDENTITY_COLS = ['male','female','transgender','other_gender','heterosexual','homosexual_gay_or_lesbian','bisexual','other_sexual_orientation','christian','jewish','muslim','hindu','buddhist','atheist','other_religion','black','white','asian','latino','other_race_or_ethnicity','physical_disability','intellectual_or_learning_disability','psychiatric_or_mental_illness','other_disability']

def build_civil_comments():
    dd = load_dataset('google/civil_comments')
    def convert(split):
        df = dd[split].to_pandas()
        out = pd.DataFrame()
        out[TEXT_COL] = df['text'].fillna('').astype(str)
        out[LABEL_COL] = np.where(df['toxicity'].astype(float) >= 0.5, 'toxic', 'non_toxic')
        existing = [c for c in CIVIL_IDENTITY_COLS if c in df.columns]
        id_sum = df[existing].fillna(0.0).astype(float).sum(axis=1) if existing else pd.Series(0, index=df.index)
        out['identity_score'] = id_sum.values
        out[DOMAIN_COL] = np.where(out['identity_score'] > 0, 'identity_mentioned', 'identity_unmentioned')
        for c in existing:
            out[f'id_{c}'] = df[c].fillna(0.0).astype(float).values
        return out[out[TEXT_COL].str.len()>0].reset_index(drop=True)
    train = convert('train'); valid = convert('validation'); test = convert('test')
    label_names = ['non_toxic', 'toxic']
    domains = ['identity_mentioned', 'identity_unmentioned']
    # Some CivilComments mirrors expose no usable identity annotation columns.
    # In that case, keep the same identity-mentioned vs identity-unmentioned
    # experiment but derive the domain from explicit identity keywords in text.
    if train[DOMAIN_COL].eq('identity_mentioned').sum() == 0:
        keyword_terms = ['woman','women','female','man','men','male','transgender','gay','lesbian','bisexual','straight','heterosexual','christian','jewish','muslim','hindu','buddhist','atheist','black','white','asian','latino','hispanic','disabled','disability','immigrant']
        pattern = r'\\b(' + '|'.join(keyword_terms) + r')\\b'
        for _df in [train, valid, test]:
            mask = _df[TEXT_COL].str.lower().str.contains(pattern, regex=True, na=False)
            _df['identity_score'] = mask.astype(float)
            _df[DOMAIN_COL] = np.where(mask, 'identity_mentioned', 'identity_unmentioned')
        print('CivilComments identity annotations unavailable or empty; using text keyword fallback for identity domain split.')
    # If annotations exist only outside the train split, rebuild train/valid/test from all annotated rows.
    domain_counts = train[DOMAIN_COL].value_counts().to_dict()
    if domain_counts.get('identity_mentioned', 0) == 0 and (valid[DOMAIN_COL].eq('identity_mentioned').any() or test[DOMAIN_COL].eq('identity_mentioned').any()):
        all_df = pd.concat([train, valid, test], ignore_index=True)
        split_frames = {}
        for d in domains:
            part = all_df[all_df[DOMAIN_COL] == d].reset_index(drop=True)
            if len(part) == 0:
                split_frames[d] = (part, part.copy(), part.copy())
                continue
            strat = part[LABEL_COL] if part[LABEL_COL].value_counts().min() >= 2 else None
            tr, tmp = train_test_split(part, test_size=0.20, random_state=SEED, stratify=strat)
            strat_tmp = tmp[LABEL_COL] if tmp[LABEL_COL].value_counts().min() >= 2 else None
            va, te = train_test_split(tmp, test_size=0.50, random_state=SEED + 1, stratify=strat_tmp)
            split_frames[d] = (tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True))
        return make_frames_from_explicit_domains(split_frames, label_names, task_name='civil_comments_toxicity')
    return make_frames_from_domain_column(train, valid, test, domains, label_names, task_name='civil_comments_toxicity')

def build_reviews_5star():
    # Yelp: 650k train, 50k test. Amazon Reviews Multi en: 200k/5k/5k in the original corpus.
    yelp = load_dataset('Yelp/yelp_review_full')
    yd_train_full = yelp['train'].to_pandas()
    yd_test_full = yelp['test'].to_pandas()
    def yelp_convert(df, split):
        out = pd.DataFrame()
        out[TEXT_COL] = df['text'].fillna('').astype(str)
        out[LABEL_COL] = df['label'].astype(int).apply(lambda x: f'{x+1}_star')
        out[DOMAIN_COL] = 'yelp_reviews'
        return out
    y_train, y_valid = train_test_split(yd_train_full, test_size=0.08, random_state=SEED, stratify=yd_train_full['label'])
    y_train, y_valid, y_test = yelp_convert(y_train, 'train'), yelp_convert(y_valid, 'valid'), yelp_convert(yd_test_full, 'test')
    # Amazon has occasional naming differences; robust fallback to Yelp split if unavailable.
    try:
        amazon = load_dataset('amazon_reviews_multi', 'en')
    except Exception as e:
        print('amazon_reviews_multi/en failed, fallback to Yelp-only domain split:', e)
        amazon = None
    if amazon is not None:
        def amz_convert(ds_split):
            df = ds_split.to_pandas()
            text_col = find_col(df.columns, ['review_body','text','content'])
            label_col = find_col(df.columns, ['stars','label'])
            out = pd.DataFrame()
            out[TEXT_COL] = df[text_col].fillna('').astype(str)
            vals = df[label_col]
            if vals.min() == 0:
                out[LABEL_COL] = vals.astype(int).apply(lambda x: f'{x+1}_star')
            else:
                out[LABEL_COL] = vals.astype(int).apply(lambda x: f'{x}_star')
            out[DOMAIN_COL] = 'amazon_reviews'
            return out
        a_train = amz_convert(amazon['train']); a_valid = amz_convert(amazon['validation']); a_test = amz_convert(amazon['test'])
    else:
        temp1, temp2, temp3 = split_train_valid_test(pd.concat([y_train, y_valid, y_test], ignore_index=True), seed=SEED)
        temp1[DOMAIN_COL]='yelp_domain_a'; temp2[DOMAIN_COL]='yelp_domain_b'; temp3[DOMAIN_COL]='yelp_domain_b'
        y_train,y_valid,y_test = temp1,temp2,temp3
        a_train,a_valid,a_test = temp2,temp3,temp3
    label_names = [f'{i}_star' for i in range(1,6)]
    return make_frames_from_explicit_domains({'yelp_reviews':(y_train,y_valid,y_test), 'amazon_reviews':(a_train,a_valid,a_test)}, label_names, task_name='reviews_5star')

def build_goemotions():
    dd = load_dataset('google-research-datasets/go_emotions', 'simplified')
    label_feature = dd['train'].features['labels'].feature
    names = getattr(label_feature, 'names', None) or [f'label_{i}' for i in range(28)]
    high = {'anger','annoyance','disapproval','fear','joy','surprise','excitement','amusement','disgust','sadness'}
    def convert(split):
        df = dd[split].to_pandas()
        rows=[]
        for _,r in df.iterrows():
            labels = r['labels']
            if labels is None or len(labels)==0: continue
            # Single-label rows are cleaner for sequence classification. Multi-label rows use the first label as a deterministic proxy.
            lid = int(labels[0])
            lname = names[lid]
            domain = 'high_arousal_emotion' if lname in high else 'low_or_neutral_emotion'
            rows.append({TEXT_COL:str(r['text']), LABEL_COL:lname, DOMAIN_COL:domain})
        return pd.DataFrame(rows)
    train=convert('train'); valid=convert('validation'); test=convert('test')
    return make_frames_from_domain_column(train, valid, test, ['high_arousal_emotion','low_or_neutral_emotion'], names, task_name='goemotions')

def build_mnli_genre():
    dd = load_dataset('nyu-mll/multi_nli')
    feat = dd['train'].features['label']
    label_names = [classlabel_name(feat, i) for i in range(3)]
    def convert(ds):
        df=ds.to_pandas()
        out=pd.DataFrame()
        out[TEXT_COL]=df['premise'].fillna('').astype(str)
        out[TEXT_PAIR_COL]=df['hypothesis'].fillna('').astype(str)
        out[LABEL_COL]=df['label'].apply(lambda x: classlabel_name(feat, int(x)))
        out[DOMAIN_COL]=df['genre'].fillna('unknown').astype(str)
        return out[out[LABEL_COL] != '-1'].reset_index(drop=True)
    train=convert(dd['train'])
    valid_m=convert(dd['validation_matched']); valid_m[DOMAIN_COL]='validation_matched'
    valid_mm=convert(dd['validation_mismatched']); valid_mm[DOMAIN_COL]='validation_mismatched'
    # choose 3 train genres with enough rows as domains
    preferred = ['fiction','government','telephone']
    domains = [d for d in preferred if d in set(train[DOMAIN_COL])]
    if len(domains)<2:
        domains = train[DOMAIN_COL].value_counts().head(3).index.tolist()
    valid = pd.concat([valid_m, valid_mm], ignore_index=True)
    test = valid.copy()
    return make_frames_from_domain_column(train, valid, test, domains, label_names, task_name='mnli_genre')

def make_frames_from_domain_column(train, valid, test, domains, label_names, task_name='task'):
    frames={}
    for d in domains:
        tr=train[train[DOMAIN_COL]==d].reset_index(drop=True)
        va=valid[valid[DOMAIN_COL]==d].reset_index(drop=True)
        te=test[test[DOMAIN_COL]==d].reset_index(drop=True)
        if len(va)==0: va=valid.sample(min(len(valid), max(500, min(len(tr)//10, 5000))), random_state=SEED).copy(); va[DOMAIN_COL]=d
        if len(te)==0: te=test.sample(min(len(test), max(500, min(len(tr)//10, 5000))), random_state=SEED+1).copy(); te[DOMAIN_COL]=d
        frames[d] = (tr, va, te)
    return {'frames': frames, 'label_names': list(label_names), 'task_name': task_name}

def make_frames_from_explicit_domains(domain_dict, label_names, task_name='task'):
    return {'frames': domain_dict, 'label_names': list(label_names), 'task_name': task_name}

def build_data():
    kind = EXPERIMENT_SPEC['task_kind']
    if kind == 'civil_toxicity_identity': return build_civil_comments()
    if kind == 'reviews_5star': return build_reviews_5star()
    if kind == 'goemotions': return build_goemotions()
    if kind == 'mnli_genre': return build_mnli_genre()
    raise ValueError(f'Unknown task_kind: {kind}')

DATA_BUNDLE = build_data()
FRAMES = DATA_BUNDLE['frames']; LABEL_NAMES = DATA_BUNDLE['label_names']
LABEL_NAMES = [str(x) for x in LABEL_NAMES]
# Remove label classes not present in the actual loaded training frames to avoid undefined metrics overload.
present_labels = sorted(set(pd.concat([v[0] for v in FRAMES.values()], ignore_index=True)[LABEL_COL].astype(str)))
if set(present_labels).issubset(set(LABEL_NAMES)):
    LABEL_NAMES = [x for x in LABEL_NAMES if x in present_labels]
label2id = {l:i for i,l in enumerate(LABEL_NAMES)}; id2label = {i:l for l,i in label2id.items()}; NUM_LABELS=len(LABEL_NAMES)
print('Task:', DATA_BUNDLE['task_name'])
print('Domains:', list(FRAMES.keys()))
print('Labels:', LABEL_NAMES[:20], '... total=', NUM_LABELS)
for d,(tr,va,te) in FRAMES.items():
    print(f'[{d}] train/valid/test:', len(tr), len(va), len(te), 'labels:', tr[LABEL_COL].value_counts().head().to_dict())

# ============================================================
# Bias / robustness evaluation data
# ============================================================
IDENTITY_TERMS = ['woman','man','Muslim','Christian','Jewish','Black person','white person','Asian person','Latino person','gay person','straight person','disabled person','immigrant','local resident']

def make_counterfactual_bias_eval(label_names):
    rows=[]
    # Use label-aware templates. This is diagnostic; it is not a substitute for real demographic evaluation.
    if set(['toxic','non_toxic']).issubset(set(label_names)):
        templates=[('The {id} gave a thoughtful and respectful comment.', 'non_toxic'), ('The {id} made a hateful and insulting comment.', 'toxic')]
    elif all(str(x).endswith('_star') for x in label_names):
        templates=[('The {id} wrote an awful review. Nothing worked and the experience was terrible.', '1_star'), ('The {id} wrote an average review. The experience was acceptable but not special.', '3_star'), ('The {id} wrote an excellent review. Everything was helpful, reliable and pleasant.', '5_star')]
        templates=[t for t in templates if t[1] in label_names]
    elif set(['entailment','neutral','contradiction']).issubset(set(label_names)):
        templates=[('Premise: A {id} is speaking at a public meeting. Hypothesis: A person is speaking.', 'entailment'), ('Premise: A {id} is speaking at a public meeting. Hypothesis: No one is speaking.', 'contradiction'), ('Premise: A {id} is speaking at a public meeting. Hypothesis: The person is a doctor.', 'neutral')]
    else:
        # GoEmotions / arbitrary label spaces: choose up to three labels with generic emotion templates.
        chosen = label_names[:min(3,len(label_names))]
        templates=[('The {id} said something that made me feel '+str(l)+'.', l) for l in chosen]
    for identity in IDENTITY_TERMS:
        for text,label in templates:
            if label not in label2id: continue
            if text.startswith('Premise:'):
                prem,hyp = text.split('Hypothesis:')
                prem = prem.replace('Premise:','').strip().format(id=identity)
                hyp = hyp.strip().format(id=identity)
                rows.append({TEXT_COL:prem, TEXT_PAIR_COL:hyp, LABEL_COL:label, GROUP_COL:identity, IDENTITY_COL:identity})
            else:
                rows.append({TEXT_COL:text.format(id=identity), LABEL_COL:label, GROUP_COL:identity, IDENTITY_COL:identity})
    return pd.DataFrame(rows)

def make_robustness_eval(base_df, limit=1000):
    base = limit_df(base_df, limit, seed=SEED)
    rows=[]
    for i,r in base.reset_index(drop=True).iterrows():
        text=str(r[TEXT_COL]); label=str(r[LABEL_COL]); pair=str(r[TEXT_PAIR_COL]) if TEXT_PAIR_COL in r and pd.notna(r.get(TEXT_PAIR_COL)) else None
        variants=[('clean', text), ('case_noise', text.upper()), ('hashtag_noise', text+' #breaking #wow'), ('emoji_noise', text+' 😂🔥'), ('url_noise', text+' https://example.com')]
        for pt,tx in variants:
            row={TEXT_COL:tx, LABEL_COL:label, PERTURB_COL:pt, SOURCE_ID_COL:i}
            if pair is not None: row[TEXT_PAIR_COL]=pair
            rows.append(row)
    return pd.DataFrame(rows)

# ============================================================
# Tokenizer/model/training utilities
# ============================================================
def load_tokenizer(model_id):
    try:
        tok=AutoTokenizer.from_pretrained(model_id, use_fast=True)
    except Exception:
        tok=AutoTokenizer.from_pretrained(model_id, use_fast=False)
    if getattr(tok, 'pad_token', None) is None:
        tok.pad_token = tok.eos_token if getattr(tok,'eos_token',None) is not None else tok.unk_token
    return tok

def df_to_ds(df, tokenizer, max_length=None):
    max_length = max_length or PROFILE['max_length']
    df=df.copy().reset_index(drop=True)
    df[TEXT_COL]=df[TEXT_COL].fillna('').astype(str)
    if TEXT_PAIR_COL in df.columns: df[TEXT_PAIR_COL]=df[TEXT_PAIR_COL].fillna('').astype(str)
    df[LABEL_COL]=df[LABEL_COL].astype(str)
    df=df[df[LABEL_COL].isin(label2id)].reset_index(drop=True)
    df['labels']=df[LABEL_COL].map(label2id).astype(int)
    keep=[TEXT_COL,'labels'] + ([TEXT_PAIR_COL] if TEXT_PAIR_COL in df.columns else [])
    ds=Dataset.from_pandas(df[keep], preserve_index=False)
    def pp(batch):
        if TEXT_PAIR_COL in batch:
            return tokenizer(batch[TEXT_COL], batch[TEXT_PAIR_COL], truncation=True, padding=False, max_length=max_length)
        return tokenizer(batch[TEXT_COL], truncation=True, padding=False, max_length=max_length)
    rem=[c for c in [TEXT_COL,TEXT_PAIR_COL] if c in ds.column_names]
    return ds.map(pp, batched=True, remove_columns=rem)

def compute_metrics_arrays(y_true, y_pred):
    ids=list(range(NUM_LABELS))
    out={
        'accuracy': accuracy_score(y_true,y_pred),
        'precision_macro': precision_score(y_true,y_pred,labels=ids,average='macro',zero_division=0),
        'recall_macro': recall_score(y_true,y_pred,labels=ids,average='macro',zero_division=0),
        'f1_macro': f1_score(y_true,y_pred,labels=ids,average='macro',zero_division=0),
        'f1_weighted': f1_score(y_true,y_pred,labels=ids,average='weighted',zero_division=0),
    }
    per=f1_score(y_true,y_pred,labels=ids,average=None,zero_division=0)
    for i,v in enumerate(per): out[f'f1_class_{id2label[i]}']=float(v)
    return {k:(float(v) if isinstance(v,(np.floating,float)) else v) for k,v in out.items()}

def compute_metrics_for_trainer(eval_pred):
    logits=eval_pred.predictions[0] if isinstance(eval_pred.predictions,tuple) else eval_pred.predictions
    return compute_metrics_arrays(eval_pred.label_ids, np.argmax(logits,axis=-1))

def build_base_model(model_id):
    model=AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
    if hasattr(model.config, 'pad_token_id') and model.config.pad_token_id is None:
        model.config.pad_token_id = 0
    if EXPERIMENT_SPEC.get('gradient_checkpointing', True):
        try: model.gradient_checkpointing_enable()
        except Exception: pass
    return model

def count_params(model):
    total=sum(p.numel() for p in model.parameters())
    train=sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train, train/total if total else 0

def choose_lora_targets(model):
    suffixes=['query_proj','value_proj','query','value','q_proj','v_proj','query_global','value_global']
    names=[n for n,_ in model.named_modules()]
    found=[]
    for s in suffixes:
        if any(n.endswith(s) for n in names): found.append(s)
    # Avoid too broad target list; use attention query/value-like modules.
    if 'query_proj' in found and 'value_proj' in found: return ['query_proj','value_proj']
    if 'query' in found and 'value' in found: return ['query','value']
    if 'q_proj' in found and 'v_proj' in found: return ['q_proj','v_proj']
    return found[:2] if found else ['query','value']

class BottleneckAdapter(nn.Module):
    def __init__(self, hidden, bottleneck=64):
        super().__init__(); self.down=nn.Linear(hidden,bottleneck); self.act=nn.GELU(); self.up=nn.Linear(bottleneck,hidden)
    def forward(self,x): return x + self.up(self.act(self.down(x)))

class AdapterWrappedModel(nn.Module):
    accepts_loss_kwargs=False
    def __init__(self, base_model, bottleneck_size=64):
        super().__init__(); self.model=base_model; self.config=base_model.config; self.accepts_loss_kwargs=False
        self.base_forward_keys=set(inspect.signature(self.model.forward).parameters.keys())
        hidden=base_model.config.hidden_size
        if hasattr(base_model,'roberta'): layers=base_model.roberta.encoder.layer
        elif hasattr(base_model,'deberta'): layers=base_model.deberta.encoder.layer
        elif hasattr(base_model,'deberta_v2'): layers=base_model.deberta_v2.encoder.layer
        elif hasattr(base_model,'longformer'): layers=base_model.longformer.encoder.layer
        elif hasattr(base_model,'bert'): layers=base_model.bert.encoder.layer
        else: raise ValueError('Adapter wrapper supports BERT/RoBERTa/DeBERTa/Longformer-like encoders only')
        for layer in layers:
            adapter=BottleneckAdapter(hidden,bottleneck_size); layer.add_module('bottleneck_adapter', adapter)
            original=layer.forward
            def make_forward(orig, ad):
                def f(*args, **kwargs):
                    kwargs.pop('num_items_in_batch',None); kwargs.pop('loss_kwargs',None)
                    outs=orig(*args, **kwargs)
                    if isinstance(outs, tuple):
                        return (ad(outs[0]),)+outs[1:]
                    return outs
                return f
            layer.forward=make_forward(original, adapter)
    def forward(self, **kwargs):
        kwargs.pop('num_items_in_batch',None); kwargs.pop('loss_kwargs',None)
        inputs={k:v for k,v in kwargs.items() if k in self.base_forward_keys and v is not None}
        return self.model(**inputs)

def build_model(model_id, method, lora_rank=None, adapter_bottleneck=None):
    base=build_base_model(model_id)
    if method=='full_ft':
        for p in base.parameters(): p.requires_grad=True
        return base
    if method=='lora':
        targets=choose_lora_targets(base); print('LoRA targets:', targets)
        cfg=LoraConfig(task_type=TaskType.SEQ_CLS, r=int(lora_rank), lora_alpha=2*int(lora_rank), lora_dropout=0.05, target_modules=targets, modules_to_save=['classifier','score'])
        model=get_peft_model(base,cfg)
        for n,p in model.named_parameters():
            if 'classifier' in n or 'score' in n: p.requires_grad=True
        if hasattr(model,'accepts_loss_kwargs'): model.accepts_loss_kwargs=False
        return model
    if method=='adapter':
        model=AdapterWrappedModel(base, int(adapter_bottleneck))
        for p in model.parameters(): p.requires_grad=False
        for n,p in model.named_parameters():
            if 'bottleneck_adapter' in n or 'classifier' in n or 'score' in n: p.requires_grad=True
        return model
    raise ValueError(method)

def filter_kwargs(callable_obj, kwargs):
    params=inspect.signature(callable_obj).parameters
    if any(p.kind==inspect.Parameter.VAR_KEYWORD for p in params.values()): return kwargs
    return {k:v for k,v in kwargs.items() if k in params}

def make_args(output_dir, method):
    lr = 6e-6 if method=='full_ft' else 1e-4
    raw=dict(output_dir=str(output_dir), num_train_epochs=PROFILE['epochs'], per_device_train_batch_size=PROFILE['batch_size'], per_device_eval_batch_size=PROFILE['eval_batch_size'], gradient_accumulation_steps=PROFILE['gradient_accumulation_steps'], learning_rate=lr, weight_decay=0.01, warmup_ratio=0.06, logging_steps=100, logging_strategy='steps', save_strategy='no', report_to=[], seed=SEED, bf16=USE_BF16, fp16=USE_FP16, tf32=torch.cuda.is_available(), remove_unused_columns=False, dataloader_num_workers=PROFILE['dataloader_num_workers'], dataloader_pin_memory=torch.cuda.is_available(), do_train=True, do_eval=True)
    params=inspect.signature(TrainingArguments.__init__).parameters
    raw['eval_strategy' if 'eval_strategy' in params else 'evaluation_strategy']='epoch'
    return TrainingArguments(**filter_kwargs(TrainingArguments.__init__, raw))

class SafeTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        inputs=dict(inputs); inputs.pop('num_items_in_batch',None); inputs.pop('loss_kwargs',None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

def make_trainer(model, args, tokenizer, train_ds=None, valid_ds=None):
    raw=dict(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds, tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics_for_trainer)
    if 'processing_class' in inspect.signature(SafeTrainer.__init__).parameters:
        raw.pop('tokenizer', None); raw['processing_class']=tokenizer
    tr=SafeTrainer(**filter_kwargs(SafeTrainer.__init__, raw))
    if hasattr(tr,'model_accepts_loss_kwargs'): tr.model_accepts_loss_kwargs=False
    return tr

def predict_df(trainer, df, tokenizer):
    ds=df_to_ds(df, tokenizer)
    out=trainer.predict(ds)
    logits=out.predictions[0] if isinstance(out.predictions,tuple) else out.predictions
    pred=np.argmax(logits,axis=-1); true=out.label_ids
    res=df.copy().reset_index(drop=True)
    res['true_id']=true; res['pred_id']=pred; res['true_label']=[id2label[int(x)] for x in true]; res['pred_label']=[id2label[int(x)] for x in pred]
    return compute_metrics_arrays(true,pred),res

def group_bias_summary(pred_df, run_meta):
    if GROUP_COL not in pred_df.columns: return None, []
    rows=[]
    for g,sub in pred_df.groupby(GROUP_COL):
        m=compute_metrics_arrays(sub.true_id.values, sub.pred_id.values); row=dict(run_meta); row[GROUP_COL]=g; row.update(m); rows.append(row)
    if not rows: return None, []
    gdf=pd.DataFrame(rows)
    pred_rate_gap=0.0
    for lab in LABEL_NAMES:
        rates=gdf.copy()
    # prediction rate by group for each predicted label
    rate_rows=[]
    for g,sub in pred_df.groupby(GROUP_COL):
        for lab in LABEL_NAMES:
            rate_rows.append({'group':g,'label':lab,'rate':float((sub.pred_label==lab).mean())})
    rdf=pd.DataFrame(rate_rows)
    if len(rdf): pred_rate_gap=float(rdf.groupby('label')['rate'].max().sub(rdf.groupby('label')['rate'].min()).max())
    summary=dict(run_meta)
    summary.update({'max_group_f1_gap':float(gdf.f1_macro.max()-gdf.f1_macro.min()), 'max_group_accuracy_gap':float(gdf.accuracy.max()-gdf.accuracy.min()), 'max_pred_rate_gap':pred_rate_gap, 'best_group':gdf.sort_values('f1_macro', ascending=False).iloc[0][GROUP_COL], 'worst_group':gdf.sort_values('f1_macro').iloc[0][GROUP_COL]})
    return summary, rows

def robustness_summary(pred_df, run_meta):
    if PERTURB_COL not in pred_df.columns: return None, []
    rows=[]
    for p,sub in pred_df.groupby(PERTURB_COL):
        m=compute_metrics_arrays(sub.true_id.values, sub.pred_id.values); row=dict(run_meta); row[PERTURB_COL]=p; row.update(m); rows.append(row)
    rdf=pd.DataFrame(rows)
    clean=float(rdf.loc[rdf[PERTURB_COL].eq('clean'),'f1_macro'].mean()) if (rdf[PERTURB_COL].eq('clean')).any() else np.nan
    pert=float(rdf.loc[~rdf[PERTURB_COL].eq('clean'),'f1_macro'].mean()) if (~rdf[PERTURB_COL].eq('clean')).any() else clean
    summ=dict(run_meta); summ.update({'clean_f1_macro':clean, 'perturbed_f1_macro':pert, 'robustness_drop':float(clean-pert) if not math.isnan(clean) else 0.0})
    return summ, rows

def counterfactual_flip_rate(pred_df, run_meta):
    if SOURCE_ID_COL not in pred_df.columns or IDENTITY_COL not in pred_df.columns: return None
    flips=[]
    for sid,sub in pred_df.groupby(SOURCE_ID_COL):
        if len(sub.pred_label.unique())>1: flips.append(1)
        else: flips.append(0)
    row=dict(run_meta); row['counterfactual_flip_rate']=float(np.mean(flips)) if flips else 0.0
    return row

# ============================================================
# Build prepared frames
# ============================================================
# Apply profile limits.
for d,(tr,va,te) in list(FRAMES.items()):
    FRAMES[d]=(limit_df(tr, PROFILE['max_train_per_domain'], seed=SEED), limit_df(va, PROFILE['max_eval_per_set'], seed=SEED+1), limit_df(te, PROFILE['max_eval_per_set'], seed=SEED+2))
base_train=pd.concat([v[0] for v in FRAMES.values()], ignore_index=True)
base_valid=pd.concat([v[1] for v in FRAMES.values()], ignore_index=True)
base_test=pd.concat([v[2] for v in FRAMES.values()], ignore_index=True)
base_train=limit_df(base_train, PROFILE['baseline_max_each_domain']*len(FRAMES) if PROFILE['baseline_max_each_domain'] else None, seed=SEED)
base_valid=limit_df(base_valid, PROFILE['max_eval_per_set']*len(FRAMES) if PROFILE['max_eval_per_set'] else None, seed=SEED+4)
base_test=limit_df(base_test, PROFILE['max_eval_per_set']*len(FRAMES) if PROFILE['max_eval_per_set'] else None, seed=SEED+5)
BIAS_EVAL = make_counterfactual_bias_eval(LABEL_NAMES)
# add source ids for CF flipping: same template id across identities is approximated by modulo template count
BIAS_EVAL[SOURCE_ID_COL] = np.arange(len(BIAS_EVAL)) % max(1, len(BIAS_EVAL)//len(IDENTITY_TERMS))
ROBUST_EVAL = make_robustness_eval(base_test, limit=min(1000, len(base_test)))
print('Prepared baseline train/valid/test:', len(base_train), len(base_valid), len(base_test))
print('Bias eval:', len(BIAS_EVAL), 'Robust eval:', len(ROBUST_EVAL))
for d,(tr,va,te) in FRAMES.items(): print(f'Prepared domain {d}:', len(tr), len(va), len(te))

# ============================================================
# Main experiment runner
# ============================================================
all_eval=[]; all_eff=[]; all_bias=[]; all_bias_detail=[]; all_rob=[]; all_rob_detail=[]; all_cf=[]; all_train_logs=[]; all_train_diag=[]

def train_one(model_id, tokenizer, run_id, domain, method, train_df, valid_df, start_ckpt, lora_rank=None, adapter_bottleneck=None, save_dir=None):
    print('\n'+'='*120); print('RUN', run_id, '| model=', model_id, '| domain=', domain, '| method=', method, '| ckpt=', start_ckpt); print('='*120)
    validate_non_empty_training_frame(train_df, run_id)
    train_ds=df_to_ds(train_df, tokenizer); valid_ds=df_to_ds(valid_df, tokenizer)
    run_meta={'run_id':run_id,'base_model':model_id,'domain':domain,'method':method,'lora_rank':lora_rank if lora_rank is not None else '', 'adapter_bottleneck':adapter_bottleneck if adapter_bottleneck is not None else ''}
    try:
        model=build_model(start_ckpt, method, lora_rank, adapter_bottleneck)
    except Exception as e:
        fail = dict(run_meta)
        fail.update({'profile': RUN_PROFILE, 'batch_size': PROFILE.get('batch_size'), 'eval_batch_size': PROFILE.get('eval_batch_size'), 'error_type': type(e).__name__, 'error': str(e)})
        pd.DataFrame([fail]).to_csv(LOG_ROOT / f'{_safe_run_filename(run_id)}__failed.csv', index=False)
        (LOG_ROOT / f'{_safe_run_filename(run_id)}__traceback.txt').write_text(format_exception(e), encoding='utf-8')
        print(f'[failed-build_model] run_id={run_id} method={method} domain={domain}: {type(e).__name__}: {e}')
        if method == 'adapter':
            print('[failed-build_model] Adapter wrapper could not attach to this model architecture. Use LoRA/full_ft for this model or set skip_failed_runs=True.')
        if EXPERIMENT_SPEC.get('skip_failed_runs', False):
            return None
        raise
    total,trainable,ratio=count_params(model); print('params:', total, 'trainable:', trainable, 'ratio:', ratio)
    args=make_args(RUN_ROOT/run_id, method)
    trainer=make_trainer(model,args,tokenizer,train_ds,valid_ds)
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    attempt = 0
    while True:
        try:
            print('[run_context]', json.dumps({**run_meta, 'profile': RUN_PROFILE, 'batch_size': args.per_device_train_batch_size, 'eval_batch_size': args.per_device_eval_batch_size}, ensure_ascii=False))
            t0=time.time(); trainer.train(); train_sec=time.time()-t0
            break
        except Exception as e:
            fail = dict(run_meta)
            fail.update({'profile': RUN_PROFILE, 'batch_size': args.per_device_train_batch_size, 'eval_batch_size': args.per_device_eval_batch_size, 'error_type': type(e).__name__, 'error': str(e)})
            pd.DataFrame([fail]).to_csv(LOG_ROOT / f'{_safe_run_filename(run_id)}__failed.csv', index=False)
            (LOG_ROOT / f'{_safe_run_filename(run_id)}__traceback.txt').write_text(format_exception(e), encoding='utf-8')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            can_retry = is_cuda_oom_like(e) and attempt < 2 and args.per_device_train_batch_size > 1
            if can_retry:
                attempt += 1
                args.per_device_train_batch_size = max(1, args.per_device_train_batch_size // 2)
                args.per_device_eval_batch_size = max(1, args.per_device_eval_batch_size // 2)
                print(f'[retry] {run_id}: CUDA/Accelerator-like error. Retrying with batch_size={args.per_device_train_batch_size}, eval_batch_size={args.per_device_eval_batch_size}')
                trainer=make_trainer(model,args,tokenizer,train_ds,valid_ds)
                continue
            print(f'[failed] run_id={run_id} method={method} domain={domain} profile={RUN_PROFILE} batch_size={args.per_device_train_batch_size} eval_batch_size={args.per_device_eval_batch_size}')
            print('[failed] For OOM, rerun with smaller profile_overrides such as batch_size=4, eval_batch_size=16.')
            if EXPERIMENT_SPEC.get('skip_failed_runs', False):
                return None
            raise
    train_log_df = extract_training_log_df(trainer, run_meta)
    train_diag = summarize_training_log_df(train_log_df)
    train_diag.update(dict(run_meta))
    all_train_diag.append(train_diag)
    if len(train_log_df):
        train_log_df.to_csv(LOG_ROOT / f'{_safe_run_filename(run_id)}__training_log.csv', index=False)
        all_train_logs.extend(train_log_df.to_dict('records'))
        print('[training]', run_id, 'first_loss=', train_diag.get('first_train_loss'), 'last_loss=', train_diag.get('last_train_loss'), 'reduction_pct=', train_diag.get('train_loss_reduction_pct'))
    peak=(torch.cuda.max_memory_allocated()/(1024**2)) if torch.cuda.is_available() else np.nan
    if save_dir:
        save_dir=Path(save_dir); shutil.rmtree(save_dir, ignore_errors=True); save_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(save_dir)); tokenizer.save_pretrained(str(save_dir))
    # evaluations
    eval_sets={'baseline_test':base_test, 'bias_eval':BIAS_EVAL, 'robustness_eval':ROBUST_EVAL}
    for d,(_,_,te) in FRAMES.items(): eval_sets[f'{d}_test']=te
    pred_cache={}
    for name,df in eval_sets.items():
        m,pred=predict_df(trainer,df,tokenizer); row=dict(run_meta); row['eval_set']=name; row.update(m); all_eval.append(row); pred_cache[name]=pred
    eff=dict(run_meta); eff.update({'total_params':int(total),'trainable_params':int(trainable),'trainable_param_ratio':float(ratio),'training_time_seconds':float(train_sec),'peak_gpu_memory_mb':float(peak) if not pd.isna(peak) else np.nan,'global_train_steps':trainer.state.global_step})
    eff.update({k:v for k,v in train_diag.items() if k not in eff})
    all_eff.append(eff)
    bs,bd=group_bias_summary(pred_cache['bias_eval'], run_meta)
    if bs: all_bias.append(bs); all_bias_detail.extend(bd)
    cf=counterfactual_flip_rate(pred_cache['bias_eval'], run_meta)
    if cf: all_cf.append(cf)
    rs,rd=robustness_summary(pred_cache['robustness_eval'], run_meta)
    if rs: all_rob.append(rs); all_rob_detail.extend(rd)
    del trainer, model; gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return save_dir

def run_all():
    model_ids=EXPERIMENT_SPEC['models']
    for model_id in model_ids:
        tokenizer=load_tokenizer(model_id)
        safe_model_name=model_id.replace('/','__')
        b0_dir=CKPT_ROOT / f'{safe_model_name}__B0'
        # B0 full FT on mixed baseline.
        train_one(model_id, tokenizer, f'{safe_model_name}__B0', 'baseline_mix', 'full_ft', base_train, base_valid, model_id, save_dir=b0_dir)
        for d,(tr,va,te) in FRAMES.items():
            if PROFILE['do_full_ft']:
                train_one(model_id, tokenizer, f'{safe_model_name}__{d}__full_ft', d, 'full_ft', tr, va, str(b0_dir))
            for r in PROFILE['lora_ranks']:
                train_one(model_id, tokenizer, f'{safe_model_name}__{d}__lora_r{r}', d, 'lora', tr, va, str(b0_dir), lora_rank=r)
            for b in PROFILE['adapter_bottlenecks']:
                train_one(model_id, tokenizer, f'{safe_model_name}__{d}__adapter_b{b}', d, 'adapter', tr, va, str(b0_dir), adapter_bottleneck=b)
    return finalize()

def finalize():
    eval_df=pd.DataFrame(all_eval); eff_df=pd.DataFrame(all_eff); bias_df=pd.DataFrame(all_bias); bias_detail_df=pd.DataFrame(all_bias_detail); robust_df=pd.DataFrame(all_rob); robust_detail_df=pd.DataFrame(all_rob_detail); cf_df=pd.DataFrame(all_cf); training_history_df=pd.DataFrame(all_train_logs); training_diag_df=pd.DataFrame(all_train_diag)
    # Baseline delta per base model and eval set
    metric_cols=['accuracy','precision_macro','recall_macro','f1_macro','f1_weighted']
    b0=eval_df[eval_df['domain'].eq('baseline_mix')][['base_model','eval_set']+metric_cols].copy().rename(columns={c:f'B0_{c}' for c in metric_cols})
    delta=eval_df[~eval_df['domain'].eq('baseline_mix')].merge(b0,on=['base_model','eval_set'],how='left')
    for c in metric_cols:
        delta[f'delta_{c}_vs_B0']=delta[c]-delta[f'B0_{c}']
    # Bias score
    if len(bias_df):
        bias_df=bias_df.merge(cf_df,on=['run_id','base_model','domain','method','lora_rank','adapter_bottleneck'],how='left') if len(cf_df) else bias_df
        bias_df['counterfactual_flip_rate']=bias_df.get('counterfactual_flip_rate',0).fillna(0)
        bias_df['bias_score']=0.35*bias_df['max_group_f1_gap'] + 0.20*bias_df['max_group_accuracy_gap'] + 0.25*bias_df['max_pred_rate_gap'] + 0.20*bias_df['counterfactual_flip_rate']
    # Ranking
    avg=eval_df[eval_df.eval_set.str.endswith('_test')].groupby(['run_id','base_model','domain','method','lora_rank','adapter_bottleneck'],dropna=False).agg(avg_test_f1_macro=('f1_macro','mean'), avg_test_accuracy=('accuracy','mean')).reset_index()
    rank=eff_df.merge(avg,on=['run_id','base_model','domain','method','lora_rank','adapter_bottleneck'],how='left')
    if len(bias_df): rank=rank.merge(bias_df[['run_id','bias_score','max_group_f1_gap','max_pred_rate_gap','counterfactual_flip_rate']],on='run_id',how='left')
    else: rank['bias_score']=0
    if len(robust_df): rank=rank.merge(robust_df[['run_id','robustness_drop']],on='run_id',how='left')
    else: rank['robustness_drop']=0
    def minmax(s, reverse=False):
        s=pd.to_numeric(s,errors='coerce').fillna(0);
        if s.max()==s.min(): out=pd.Series(1.0,index=s.index)
        else: out=(s-s.min())/(s.max()-s.min())
        return 1-out if reverse else out
    rank['norm_f1']=minmax(rank['avg_test_f1_macro'])
    rank['norm_eff']=0.5*minmax(rank['trainable_param_ratio'],reverse=True)+0.5*minmax(rank['training_time_seconds'],reverse=True)
    rank['norm_bias']=minmax(rank['bias_score'],reverse=True)
    rank['norm_robust']=minmax(rank['robustness_drop'],reverse=True)
    rank['composite_score']=0.40*rank['norm_f1']+0.20*rank['norm_eff']+0.25*rank['norm_bias']+0.15*rank['norm_robust']
    target=EXPERIMENT_SPEC.get('target_bias_score',0.05)
    rank['target_bias_distance']=(rank['bias_score']-target).abs()
    rank['target_bias_recommendation_score']=0.55*rank['norm_f1']+0.25*(1-minmax(rank['target_bias_distance']))+0.20*rank['norm_eff']
    ranking=rank.sort_values('composite_score',ascending=False).reset_index(drop=True)
    target_rec=rank.sort_values('target_bias_recommendation_score',ascending=False).reset_index(drop=True)
    # Save
    # Loss/learning-progress visualizations.
    fig_paths = plot_training_diagnostics(training_history_df, training_diag_df, ranking)
    tables={'all_eval_metrics':eval_df,'efficiency':eff_df,'training_history':training_history_df,'training_diagnostics':training_diag_df,'delta_vs_baseline':delta,'bias_summary':bias_df,'bias_group_detail':bias_detail_df,'counterfactual_flip':cf_df,'robustness_summary':robust_df,'robustness_detail':robust_detail_df,'final_ranking':ranking,'target_bias_recommendations':target_rec,'bias_modeling_dataset':rank}
    for name,df in tables.items(): df.to_csv(TABLE_ROOT/f'{name}.csv',index=False)
    # Markdown report
    best=ranking.iloc[0].to_dict() if len(ranking) else {}
    report=f"""# {EXPERIMENT_SPEC['title']}\n\n- Task kind: {EXPERIMENT_SPEC['task_kind']}\n- Run profile: {RUN_PROFILE}\n- Device: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)\n- Models: {', '.join(EXPERIMENT_SPEC['models'])}\n- Domains: {', '.join(FRAMES.keys())}\n- Labels: {len(LABEL_NAMES)} classes\n\n## Best composite run\n\n```json\n{json.dumps(best, ensure_ascii=False, indent=2, default=str)}\n```\n\n## Training progress / loss diagnostics\n\nThe notebook now saves Trainer log history and learning curves. Check:\n\n- `tables/training_history.csv`: raw Trainer `log_history` rows for every run.\n- `tables/training_diagnostics.csv`: compact loss reduction, best validation metric, and overfitting-gap summary.\n- `figures/training_loss_by_run.png`\n- `figures/validation_loss_by_run.png`\n- `figures/validation_f1_macro_by_run.png`\n- `figures/top_training_loss_reduction_pct.png`\n\nGenerated figure files:\n\n```json\n{json.dumps(fig_paths, ensure_ascii=False, indent=2, default=str)}\n```\n\n## Interpretation\n\n- Training loss shows whether optimization is actually moving. A clear downward trend means the model is fitting the training data.\n- Validation loss and validation F1 show whether that learning transfers to held-out data.\n- A growing gap between validation loss and training loss can indicate overfitting.\n- `bias_score` combines group F1 gap, group accuracy gap, prediction-rate gap, and counterfactual flip rate. Lower means less observed bias on the current evaluation set, not guaranteed absence of bias.\n- `target_bias_recommendations.csv` ranks settings by closeness to TARGET_BIAS_SCORE plus performance and efficiency.\n"""
    (OUT_ROOT/'report.md').write_text(report,encoding='utf-8')
    zip_path=shutil.make_archive(str(OUT_ROOT),'zip',OUT_ROOT)
    print('Saved outputs to', OUT_ROOT)
    print('Zip:', zip_path)
    print('Top final ranking:'); safe_display(ranking, 20)
    print('Top target bias recommendations:'); safe_display(target_rec, 20)
    return tables

RESULTS = run_all()


ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)